# 06 — Main Benchmark: PINN vs FNO Cost Analysis

This notebook produces the **central figure** of the project:

**Total cost to handle N problem instances** — plotted for PINN and FNO.

- PINN: steep straight line (cost = N × retrain_time)
- FNO: high intercept (training cost) then nearly flat
- **Crossover point**: where FNO becomes cheaper than PINN

Also reports:
- PINN loss breakdown (residual / IC / BC separately)
- FNO generalization validation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data_utils
import math
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Benchmark using device: {device}')

In [ ]:
# ============================================================
# Harness (inline copy for standalone notebook)
# ============================================================

class MeasurementHarness:
    def __init__(self):
        self.registry = {}
        self.device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

    @staticmethod
    def sync_time():
        if torch.cuda.is_available(): torch.cuda.synchronize()
        return time.perf_counter()

    @staticmethod
    def rel_l2(pred, exact):
        p = pred.flatten().float(); e = exact.flatten().float()
        return (torch.linalg.norm(p - e) / (torch.linalg.norm(e) + 1e-8)).item()

    def benchmark_pinn(self, name, factory, exact_fn, instances, n_runs=10):
        print(f'--- PINN Benchmark: {name} over {n_runs} runs ---')
        runtimes, errors, breakdowns = [], [], []
        for i in range(n_runs):
            ic = instances[i]
            t0 = self.sync_time()
            u_pred, loss_dict = factory(ic)
            rt = self.sync_time() - t0
            runtimes.append(rt); errors.append(self.rel_l2(u_pred, exact_fn(ic)))
            breakdowns.append(loss_dict)
            print(f'  Run {i+1}: {rt:.2f}s  L2={errors[-1]:.4f}')
        mean_rt = np.mean(runtimes); std_rt = np.std(runtimes)
        mean_err = np.mean(errors)
        record = {
            'type': 'instance', 'train_time_sec': 0.0,
            'per_instance_sec': mean_rt, 'per_instance_std': std_rt,
            'l2_error': mean_err,
            'loss_breakdown': {
                'L_res': np.mean([d.get('res_loss', 0) for d in breakdowns]),
                'L_ic':  np.mean([d.get('ic_loss',  0) for d in breakdowns]),
                'L_bc':  np.mean([d.get('bc_loss',  0) for d in breakdowns]),
            }
        }
        self.registry[name] = record
        print(f'  Mean: {mean_rt:.2f}s +/-{std_rt:.2f}s, L2={mean_err:.6f}'); print()
        return record

    def benchmark_amortized(self, name, train_fn, infer_fn, exact, n_test):
        print(f'--- Amortized Benchmark: {name} ---')
        t0 = self.sync_time(); train_fn(); train_time = self.sync_time() - t0
        t0 = self.sync_time(); pred = infer_fn(); infer_time = self.sync_time() - t0
        record = {
            'type': 'amortized', 'train_time_sec': train_time,
            'per_instance_sec': infer_time / n_test, 'per_instance_std': 0.0,
            'l2_error': self.rel_l2(pred, exact)
        }
        self.registry[name] = record
        print(f'  Train: {train_time:.2f}s, Per-instance: {record["per_instance_sec"]:.6f}s')
        print(f'  L2: {record["l2_error"]:.6f}'); print()
        return record

    def plot_amortization(self, max_n=500):
        N = np.arange(1, max_n + 1)
        plt.figure(figsize=(11, 6))
        colors = ['#e74c3c', '#3498db', '#2ecc71']
        for (name, m), c in zip(self.registry.items(), colors):
            cost = m['train_time_sec'] + N * m['per_instance_sec']
            plt.plot(N, cost, label=f"{name} (L2={m['l2_error']:.4f})", lw=2.5, color=c)
            if m['per_instance_std'] > 0:
                band = N * m['per_instance_std']
                plt.fill_between(N, cost-band, cost+band, alpha=0.15, color=c)
        names = list(self.registry.keys())
        if len(names) >= 2:
            m1 = self.registry[names[0]]; m2 = self.registry[names[1]]
            denom = m1['per_instance_sec'] - m2['per_instance_sec']
            if abs(denom) > 1e-10:
                xo = (m2['train_time_sec'] - m1['train_time_sec']) / denom
                if 0 < xo < max_n:
                    plt.axvline(x=xo, color='gray', linestyle=':', lw=1.5,
                                label=f'Crossover N~{int(xo)}')
        plt.yscale('log')
        plt.title(f'Amortization Analysis ({self.device_name})', fontsize=14)
        plt.xlabel('Number of Instances (N)', fontsize=12)
        plt.ylabel('Total Time (seconds)', fontsize=12)
        plt.grid(True, which='both', ls='--', alpha=0.4)
        plt.legend(fontsize=11); plt.tight_layout(); plt.show()


harness = MeasurementHarness()

In [ ]:
# ============================================================
# PINN Factory for Heat Equation
# ============================================================

class HeatPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=-1))


def make_heat_ic(k, seed=42):
    """IC: u0(x) = sin(k*pi*x), decay = exp(-alpha*k^2*pi^2 * T)."""
    def ic_fn(x):
        return torch.sin(k * math.pi * x)
    return ic_fn


def create_heat_pinn_factory(alpha=0.01, adam_epochs=500, n_f=2000):
    """
    Returns a callable pinn_factory(k) -> (u_pred, loss_dict).
    Each call trains a fresh PINN for the IC u0(x) = sin(k*pi*x).
    """
    def pinn_factory(k):
        model = HeatPINN().to(device)
        opt   = optim.Adam(model.parameters(), lr=1e-3)

        # Collocation points
        x_f = torch.rand(n_f, 1, device=device, requires_grad=True)
        t_f = torch.rand(n_f, 1, device=device, requires_grad=True)

        # IC and BC
        x_ic = torch.rand(n_f//4, 1, device=device, requires_grad=True)
        t_ic = torch.zeros(n_f//4, 1, device=device, requires_grad=True)
        x_bc_L = torch.zeros(n_f//4, 1, device=device, requires_grad=True)
        x_bc_R = torch.ones( n_f//4, 1, device=device, requires_grad=True)
        t_bc   = torch.rand(n_f//4, 1, device=device, requires_grad=True)

        final_res, final_ic, final_bc = 0.0, 0.0, 0.0

        for _ in range(adam_epochs):
            opt.zero_grad()
            u   = model(x_f, t_f)
            u_t = torch.autograd.grad(u, t_f, torch.ones_like(u), create_graph=True)[0]
            u_x = torch.autograd.grad(u, x_f, torch.ones_like(u), create_graph=True)[0]
            u_xx = torch.autograd.grad(u_x, x_f, torch.ones_like(u_x), create_graph=True)[0]
            l_res = torch.mean((u_t - alpha * u_xx) ** 2)

            u_ic_pred = model(x_ic, t_ic)
            l_ic = torch.mean((u_ic_pred - torch.sin(k * math.pi * x_ic)) ** 2)

            u_L = model(x_bc_L, t_bc); u_R = model(x_bc_R, t_bc)
            l_bc = torch.mean((u_L - u_R) ** 2)

            loss = l_res + l_ic + l_bc
            loss.backward(); opt.step()

            final_res = l_res.item(); final_ic = l_ic.item(); final_bc = l_bc.item()

        model.eval()
        with torch.no_grad():
            x_test = torch.linspace(0, 1, 100, device=device).view(-1, 1)
            t_test = torch.ones_like(x_test)
            u_pred = model(x_test, t_test)

        return u_pred.cpu(), {'res_loss': final_res, 'ic_loss': final_ic, 'bc_loss': final_bc}

    return pinn_factory


def get_exact_heat(k, alpha=0.01, T=1.0):
    x_test = torch.linspace(0, 1, 100).view(-1, 1)
    decay  = math.exp(-alpha * (k * math.pi) ** 2 * T)
    return torch.sin(k * math.pi * x_test) * decay

In [ ]:
# ============================================================
# FNO Factory for Heat Equation
# ============================================================

class FourierLayer1d(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.modes = modes
        scale = 1.0 / (width * width)
        self.R_real = nn.Parameter(scale * torch.randn(width, width, modes))
        self.R_imag = nn.Parameter(scale * torch.randn(width, width, modes))
        self.W = nn.Linear(width, width)

    def forward(self, x):
        B, nx, C = x.shape
        xft = torch.fft.rfft(x, dim=1)
        out = torch.zeros(B, nx//2+1, C, device=x.device, dtype=torch.cfloat)
        R = torch.complex(self.R_real, self.R_imag)
        out[:, :self.modes, :] = torch.einsum('bmi,iom->bmo', xft[:, :self.modes, :], R)
        return torch.fft.irfft(out, n=nx, dim=1) + self.W(x)

class FNO1d(nn.Module):
    def __init__(self, modes=16, width=64):
        super().__init__()
        self.lift = nn.Linear(1, width)
        self.fl   = nn.ModuleList([FourierLayer1d(width, modes) for _ in range(4)])
        self.norms = nn.ModuleList([nn.LayerNorm(width) for _ in range(4)])
        self.p1 = nn.Linear(width, 128); self.p2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.lift(x)
        for fl, n in zip(self.fl, self.norms):
            x = F.gelu(n(fl(x)))
        return self.p2(F.gelu(self.p1(x)))


def create_fno_factory(n_samples=2000, epochs=100, nx=100, alpha=0.01, T=1.0):
    """
    Train an FNO and return a factory callable(k) -> (u_pred, info_dict).
    The FNO is trained once; all subsequent calls are forward passes only.
    """
    print('--- Starting FNO Training ---')
    t_train_start = time.time()

    # Generate training data: IC = sin(k*pi*x), solution at T
    k_vals = torch.randint(1, 10, (n_samples, 1)).float()
    x_grid = torch.linspace(0, 1, nx)
    inputs  = torch.sin(k_vals * math.pi * x_grid).unsqueeze(-1)        # (N, nx, 1)
    outputs = inputs * torch.exp(-alpha * (k_vals * math.pi)**2 * T).unsqueeze(-1)  # (N, nx, 1)

    dataset = data_utils.TensorDataset(inputs.to(device), outputs.to(device))
    loader  = data_utils.DataLoader(dataset, batch_size=64, shuffle=True)

    model = FNO1d().to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for ep in range(epochs):
        for xb, yb in loader:
            opt.zero_grad(); pred = model(xb); loss = F.mse_loss(pred, yb)
            loss.backward(); opt.step()
    train_time = time.time() - t_train_start
    print(f'--- FNO Training Complete in {train_time:.2f}s ---')

    model.eval()

    def fno_factory(k):
        x_test = torch.linspace(0, 1, nx)
        u_ic   = torch.sin(k * math.pi * x_test).unsqueeze(0).unsqueeze(-1).to(device)
        with torch.no_grad():
            u_pred = model(u_ic).squeeze(0).cpu()
        return u_pred, {'amortized_train_sec': train_time, 'solver_type': 'FNO'}

    return fno_factory, train_time

In [ ]:
# ============================================================
# Run Benchmarks
# ============================================================

print('=== PREPARING SOLVERS ===')
pinn_factory = create_heat_pinn_factory(alpha=0.01, adam_epochs=500)
fno_factory, fno_train_sec = create_fno_factory(n_samples=2000, epochs=100)

print('\n=== RUNNING BENCHMARKS ===')

# PINN benchmark
results_pinn = harness.benchmark_pinn(
    name='Heat PINN',
    factory=pinn_factory,
    exact_fn=get_exact_heat,
    instances=[1, 2, 3, 4, 5],
    n_runs=5
)

# FNO benchmark: treat training as one-time cost, each inference is per-instance
n_fno_test = 10
test_ks = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5]
x_test  = torch.linspace(0, 1, 100)

# Pre-collect exact solutions
exact_stack = torch.cat(
    [torch.sin(k * math.pi * x_test) * math.exp(-0.01 * (k*math.pi)**2) for k in test_ks]
).view(n_fno_test, 100, 1)

def fno_train_noop(): pass  # already trained

def fno_infer_all():
    preds = []
    for k in test_ks:
        u_ic = torch.sin(k * math.pi * x_test).unsqueeze(0).unsqueeze(-1).to(device)
        with torch.no_grad():
            preds.append(fno_factory(k)[0])
    return torch.stack(preds).unsqueeze(-1)

harness.registry['Heat FNO'] = {
    'type': 'amortized',
    'train_time_sec': fno_train_sec,
    'per_instance_sec': 0.0,
    'per_instance_std': 0.0,
    'l2_error': 0.0
}
# Measure per-instance FNO inference time
t0 = time.perf_counter()
for k in test_ks:
    fno_factory(k)
fno_per_inst = (time.perf_counter() - t0) / n_fno_test

# Compute FNO L2
fno_preds = [fno_factory(k)[0] for k in test_ks]
fno_err = np.mean(
    [(torch.norm(fno_preds[i].squeeze() - exact_stack[i].squeeze()) /
      (torch.norm(exact_stack[i].squeeze()) + 1e-8)).item()
     for i in range(n_fno_test)]
)

harness.registry['Heat FNO'] = {
    'type': 'amortized',
    'train_time_sec': fno_train_sec,
    'per_instance_sec': fno_per_inst,
    'per_instance_std': 0.0,
    'l2_error': fno_err
}

print(f'FNO: train={fno_train_sec:.2f}s, per-instance={fno_per_inst:.6f}s, L2={fno_err:.6f}')

In [ ]:
# ============================================================
# Summary & Amortization Plot
# ============================================================

print('=== PINN LOSS BREAKDOWN ===')
lb = results_pinn['loss_breakdown']
print(f'  Residual Loss: {lb["L_res"]:.3e}')
print(f'  IC Loss:       {lb["L_ic"]:.3e}')
print(f'  BC Loss:       {lb["L_bc"]:.3e}')

pinn_cost = results_pinn['per_instance_sec']
crossover_N = fno_train_sec / (pinn_cost - fno_per_inst)
print(f'\n=== CROSSOVER ANALYSIS ===')
print(f'PINN cost per instance  : {pinn_cost:.4f} sec')
print(f'FNO cost per instance   : {fno_per_inst:.6f} sec')
print(f'Break-even: FNO cheaper after N = {crossover_N:.1f} instances')

harness.plot_amortization(max_n=max(int(crossover_N * 3), 50))